## 03 - Training

**Imports and paths**

In [ ]:
import os
from pathlib import Path

import joblib
import pandas as pd
import sklearn
import xgboost
import yaml
from dotenv import load_dotenv
from ruamel.yaml import YAML
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

import wandb

PROJECT_ROOT = Path("..").resolve()
CONFIG_PATH = PROJECT_ROOT / "config.yml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

TARGET = cfg["data"]["target"]
FEATURES = cfg["features"]
ENCODERS_DIR = PROJECT_ROOT / cfg["artifacts"]["encoders_dir"]
MODELS_DIR = PROJECT_ROOT / cfg["artifacts"]["models_dir"]
METRICS_PATH = PROJECT_ROOT / cfg["artifacts"]["metrics_path"]

# One versioned file for all preprocessing objects and one for all trained models
VERSION = cfg["experiment"]["version"]
PREPROCESSORS_PATH = ENCODERS_DIR / f"preprocessors_{VERSION}.joblib"
MODELS_PATH = MODELS_DIR / f"models_{VERSION}.joblib"

# Shared W&B settings; run files go to <project root>/wandb instead of the notebook folder
WANDB_SETTINGS = {
    "project": cfg["wandb"]["project"],
    "mode": cfg["wandb"]["mode"],
    "dir": PROJECT_ROOT / cfg["wandb"]["dir"],
    "group": VERSION,
}

**3.1 Load cleaned dataset**

In [2]:
# Identifiers and the raw visit date are dropped; they carry no generalisable signal for a new patient
df = pd.read_csv(PROJECT_ROOT / cfg["data"]["cleaned_path"])
df = df.drop(columns=cfg["data"]["drop_columns"])
print(df.shape)
df.head()

(4974, 14)


,Age,Gender,City,BMI,Blood_Pressure,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Follow_Up,Diagnosis_Code,Notes,Has_Disease
0,25.0,F,Boston,22.000000,120/80,83.0,190.0,N,Y,none,30.0,A00,Patient feeling fine 🙂,0
1,25.0,M,Los Angeles,30.100000,120/80,85.0,125.0,N,N,none,14.0,I10,Patient feeling fine 🙂,1
2,20.0,F,New York,22.000000,120/80,63.0,125.0,N,Ex,none,30.0,E11.9,No Notes,0
3,25.0,F,Boston,23.075909,120/80,85.0,250.0,N,Ex,"aspirin,metformin",30.0,Unknown,No Notes,0
4,20.0,F,Chicago,21.500000,120/80,85.0,250.0,N,Ex,none,30.0,A00,"BP high, needs recheck",1


**3.2 Train/test split**

In [3]:
# Stratify to keep the same class balance in both splits; the fixed seed makes the split reproducible
X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=cfg["data"]["test_size"],
    stratify=y,
    random_state=cfg["data"]["random_state"],
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (3979, 13) | Test: (995, 13)


**3.3 Fit encoders and scaler on train**

In [4]:
# Every encoder is fitted on training rows only, so no information from the test set leaks into the features

# Nominal categories have no order, so each value gets its own 0/1 column; unseen values at inference become all zeros
one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
one_hot_encoder.fit(X_train[FEATURES["one_hot"]])

# Blood_Pressure has ~850 distinct readings, too many for one-hot, so each reading is encoded by how often it occurs
frequency_maps = {col: X_train[col].value_counts(normalize=True).to_dict() for col in FEATURES["frequency"]}

# Follow_Up has two ordered values (14, 30 days); OrdinalEncoder is the feature-side equivalent of LabelEncoder
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
ordinal_encoder.fit(X_train[FEATURES["ordinal"]])

# Put numeric features on one scale so distance and weight based models (SVM, LR) are not dominated by large values
scaler = StandardScaler()
scaler.fit(X_train[FEATURES["scale"]])

,copy,True
,with_mean,True
,with_std,True


**3.4 Transform train and test**

In [5]:
def transform_features(X: pd.DataFrame) -> pd.DataFrame:
    """Apply the fitted encoders and scaler; inference must apply exactly the same steps."""
    scaled = pd.DataFrame(scaler.transform(X[FEATURES["scale"]]), columns=FEATURES["scale"], index=X.index)
    ordinal = pd.DataFrame(ordinal_encoder.transform(X[FEATURES["ordinal"]]), columns=FEATURES["ordinal"], index=X.index)
    # Readings never seen in training get frequency 0
    frequency = pd.DataFrame(
        {f"{col}_freq": X[col].map(frequency_maps[col]).fillna(0.0) for col in FEATURES["frequency"]},
        index=X.index,
    )
    one_hot = pd.DataFrame(
        one_hot_encoder.transform(X[FEATURES["one_hot"]]),
        columns=one_hot_encoder.get_feature_names_out(),
        index=X.index,
    )
    return pd.concat([scaled, ordinal, frequency, one_hot], axis=1)


X_train_enc = transform_features(X_train)
X_test_enc = transform_features(X_test)
print(f"Encoded train: {X_train_enc.shape} | Encoded test: {X_test_enc.shape}")
X_train_enc.head()

Encoded train: (3979, 29) | Encoded test: (995, 29)


,Age,BMI,Heart_Rate,Cholesterol_Level,Follow_Up,Blood_Pressure_freq,Gender_F,Gender_M,City_Boston,City_Chicago,...,Diagnosis_Code_Unknown,"Notes_BP high, needs recheck",Notes_Follow up soon,Notes_No Notes,Notes_Patient feeling fine 🙂,Smoker_Ex,Smoker_N,Smoker_Y,Diabetic_N,Diabetic_Y
4620,-0.345009,-0.190354,-0.484184,0.762185,0.0,0.798191,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
61,-0.449432,-0.000939,-1.652629,-0.436405,1.0,0.000503,1.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4891,-1.441447,-0.354804,0.976371,-0.436405,0.0,0.000251,1.0,0.0,0.0,1.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
887,-0.188375,0.500333,0.002667,0.762185,0.0,0.798191,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
4014,-0.449432,-0.000939,0.002667,-1.734878,1.0,0.798191,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


**3.5 Save preprocessors**

In [6]:
ENCODERS_DIR.mkdir(parents=True, exist_ok=True)

# All fitted preprocessing objects in one file, so inference always loads a matching set
preprocessors = {
    "one_hot_encoder": one_hot_encoder,
    "frequency_maps": frequency_maps,
    "ordinal_encoder": ordinal_encoder,
    "standard_scaler": scaler,
    # Column order the models were trained on; inference input must match it exactly
    "feature_columns": list(X_train_enc.columns),
}
joblib.dump(preprocessors, PREPROCESSORS_PATH)
print(f"Saved {list(preprocessors)} to {PREPROCESSORS_PATH.name}")

Saved ['one_hot_encoder', 'frequency_maps', 'ordinal_encoder', 'standard_scaler', 'feature_columns'] to preprocessors_v1.joblib


**3.6 Weights & Biases login**

In [7]:
# Load the API key from .env (ignored by git) so it is never hard-coded in the notebook or config
load_dotenv(PROJECT_ROOT / cfg["wandb"]["env_file"])

# Team/username also comes from .env; read it only after the file is loaded
WANDB_SETTINGS["entity"] = os.getenv(cfg["wandb"]["entity_env"])

if cfg["wandb"]["mode"] == "online":
    api_key = os.getenv(cfg["wandb"]["api_key_env"])
    if not api_key:
        raise ValueError(f"Set {cfg['wandb']['api_key_env']} in {cfg['wandb']['env_file']} or change wandb.mode in config.yml")
    wandb.login(key=api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/nipuna/.netrc
wandb: Currently logged in as: tennakoon1997 (tennakoon1997-nipunadev) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


**3.7 Train and evaluate models**

In [8]:
MODEL_CLASSES = {
    "random_forest": RandomForestClassifier,
    "logistic_regression": LogisticRegression,
    "svm": SVC,
    "xgboost": XGBClassifier,
}


def evaluate(model, X: pd.DataFrame, y: pd.Series) -> dict:
    pred = model.predict(X)
    # ROC-AUC only needs a ranking score; SVC's predict_proba comes from an internal CV calibration
    # that can disagree with its own predictions, so prefer decision_function when the model has one
    scores = model.decision_function(X) if hasattr(model, "decision_function") else model.predict_proba(X)[:, 1]
    return {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, scores),
    }

In [9]:
results = []
trained_models = {}

for name, spec in cfg["models"].items():
    model = MODEL_CLASSES[name](**spec["params"])
    model.fit(X_train_enc, y_train)

    train_metrics = evaluate(model, X_train_enc, y_train)
    test_metrics = evaluate(model, X_test_enc, y_test)
    trained_models[name] = {"version": spec["version"], "model": model}

    # One W&B run per model, grouped by experiment version, so train vs test metrics can be compared side by side
    run = wandb.init(
        **WANDB_SETTINGS,
        name=f"{name}-v{spec['version']}",
        job_type="train",
        config={
            "model": name,
            "model_version": spec["version"],
            "params": spec["params"],
            "features": FEATURES,
            "n_encoded_features": X_train_enc.shape[1],
            "train_rows": len(X_train_enc),
            "test_rows": len(X_test_enc),
            "sklearn_version": sklearn.__version__,
            "xgboost_version": xgboost.__version__,
        },
    )
    run.log({
        **{f"train/{k}": v for k, v in train_metrics.items()},
        **{f"test/{k}": v for k, v in test_metrics.items()},
        "test/confusion_matrix": wandb.plot.confusion_matrix(
            y_true=y_test.tolist(),
            preds=model.predict(X_test_enc).tolist(),
            class_names=["no disease", "disease"],
        ),
    })
    run.finish()

    results.append({
        "model": name,
        "version": spec["version"],
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"test_{k}": v for k, v in test_metrics.items()},
    })
    print(f"{name}: test accuracy {test_metrics['accuracy']:.3f}, test ROC-AUC {test_metrics['roc_auc']:.3f}")

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


test/accuracy,▁
test/f1,▁
test/precision,▁
test/recall,▁
test/roc_auc,▁
train/accuracy,▁
train/f1,▁
train/precision,▁
train/recall,▁
train/roc_auc,▁
test/accuracy,0.50854


random_forest: test accuracy 0.509, test ROC-AUC 0.512


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


test/accuracy,▁
test/f1,▁
test/precision,▁
test/recall,▁
test/roc_auc,▁
train/accuracy,▁
train/f1,▁
train/precision,▁
train/recall,▁
train/roc_auc,▁
test/accuracy,0.48643


...

test/accuracy,▁
test/f1,▁
test/precision,▁
test/recall,▁
test/roc_auc,▁
train/accuracy,▁
train/f1,▁
train/precision,▁
train/recall,▁
train/roc_auc,▁
test/accuracy,0.50251


svm: test accuracy 0.503, test ROC-AUC 0.492


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


test/accuracy,▁
test/f1,▁
test/precision,▁
test/recall,▁
test/roc_auc,▁
train/accuracy,▁
train/f1,▁
train/precision,▁
train/recall,▁
train/roc_auc,▁
test/accuracy,0.49849


xgboost: test accuracy 0.498, test ROC-AUC 0.482


**3.8 Save models**

In [10]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# All trained models in one versioned file; each entry keeps its own model version from config.yml
joblib.dump(trained_models, MODELS_PATH)
print(f"Saved {list(trained_models)} to {MODELS_PATH.name}")

# Log both files as W&B artifacts, tagged with the experiment version, so each run is traceable to its exact files
run = wandb.init(**WANDB_SETTINGS, name=f"artifacts-{VERSION}", job_type="upload")
for path, artifact_type in [(PREPROCESSORS_PATH, "preprocessor"), (MODELS_PATH, "model")]:
    artifact = wandb.Artifact(path.stem.removesuffix(f"_{VERSION}"), type=artifact_type, metadata={"experiment_version": VERSION})
    artifact.add_file(str(path))
    run.log_artifact(artifact, aliases=[VERSION])
run.finish()

Saved ['random_forest', 'logistic_regression', 'svm', 'xgboost'] to models_v1.joblib


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


**3.9 Model comparison**

In [11]:
comparison = pd.DataFrame(results).set_index("model").round(4)
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
comparison.to_csv(METRICS_PATH)
comparison

,version,train_accuracy,train_precision,train_recall,train_f1,train_roc_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
model,,,,,,,,,,,
random_forest,1.0.0,0.8110,0.8122,0.8040,0.8081,0.8982,0.5085,0.5034,0.4553,0.4781,0.5124
logistic_regression,1.0.0,0.5147,0.5109,0.4540,0.4808,0.5283,0.4864,0.4782,0.4228,0.4488,0.4840
svm,1.0.0,0.6725,0.6709,0.6638,0.6673,0.7343,0.5025,0.4969,0.4939,0.4954,0.4921
xgboost,1.0.0,0.7175,0.7155,0.7125,0.7140,0.8089,0.4985,0.4927,0.4817,0.4872,0.4821


**3.10 Select best model**

In [12]:
# Pick the model with the highest selection metric and record it in config.yml, so inference loads it automatically
metric = cfg["experiment"]["selection_metric"]
best_name = comparison[metric].idxmax()
best_version = str(comparison.loc[best_name, "version"])

# Round-trip YAML keeps the comments and key order of config.yml when writing it back
yaml_rt = YAML()
config_doc = yaml_rt.load(CONFIG_PATH)
config_doc["experiment"]["best_model"]["name"] = best_name
config_doc["experiment"]["best_model"]["version"] = best_version
yaml_rt.dump(config_doc, CONFIG_PATH)

print(f"Best model by {metric}: {best_name} v{best_version} ({comparison.loc[best_name, metric]:.4f}), saved to config.yml")

Best model by test_roc_auc: random_forest v1.0.0 (0.5124), saved to config.yml
